# Live exercise: make it robust, then prove it

**Part 1, `try` / `except`.** Three small functions, each catching one named
error. No bare `except:`.

**Part 2, tests.** At least two tests per function, including the awkward
case. First as functions you run here, then as a real test file that
`python -m pytest` runs.

The same exercise exists as plain files, `exercises/robust.py` and
`exercises/test_robust.py`, if you would rather work in VS Code directly.

In [ ]:
import csv
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

PLAYS = "data/clean/plays.csv"

## Given: functions you already have

The session 3 statistics functions and two helpers from the walkthrough.
Run this cell; do not change it.

In [ ]:
def total(numbers):
    """Return the sum of a list of numbers."""
    running = 0
    for number in numbers:
        running += number
    return running


def average(numbers):
    """Return the mean of a list of numbers."""
    return total(numbers) / len(numbers)


def highest(numbers):
    """Return the largest value in a list of numbers."""
    best = numbers[0]
    for number in numbers:
        if number > best:
            best = number
    return best


def load_rows(path):
    """Return the CSV at path as a list of dictionaries, one per row."""
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def minutes_on(rows, device):
    """Return the minutes_played of every play on one device, as numbers."""
    return [float(row["minutes_played"]) for row in rows if row["device"] == device]

## Part 1

### 1. `safe_highest`

`safe_highest(numbers)` returns the largest value, or `None` for an empty
list. Do not copy the loop: call `highest`, and catch the error it raises
on an empty list.

Which line of `highest` raises it, and which error is it? Try
`highest([])` first and read the message.

In [ ]:
# TODO: def safe_highest(numbers):


# print(safe_highest([3, 10, 7]))     # 10
# print(safe_highest([]))             # None

### 2. `to_minutes`

The messy file from session 9 has minutes like `"4.65"`, `"4.2 min"` and
`""`. `to_minutes(text)` returns a float when there is a number in there,
and `None` when there is not.

Strip the spaces, take `" min"` off the end if it is there
(`text.endswith(" min")` and a slice), then try `float()`.

In [ ]:
# TODO: def to_minutes(text):


# print(to_minutes("4.65"), to_minutes("4.2 min"), to_minutes(""))

Then run it over the whole minutes column of the messy file. How many
values become numbers, and how many `None`? Compare with the walkthrough,
where plain `to_number` managed 2,033 and gave up on 190.

In [ ]:
with open("data/messy/plays_messy.csv", encoding="utf-8") as f:
    raw = [row["minutes played "] for row in csv.DictReader(f)]

# TODO: count how many to_minutes(text) gives a number for, and how many None

### 3. `summary_line`, the outer layer

`summary_line(path, device)` always returns one sentence, whatever goes
wrong:

| Call | Returns |
|---|---|
| `summary_line(PLAYS, "phone")` | `"1039 plays on the phone, 4.15 minutes on average"` |
| `summary_line(PLAYS, "radio")` | `"No plays on the radio, so there is no average."` |
| `summary_line("data/clean/play.csv", "phone")` | `"No file at data/clean/play.csv. ..."` |

Two things can go wrong, so use two `try` blocks, each with its own named
`except`. Use `load_rows`, `minutes_on` and `average` as they are; do not
add any `try` to them.

In [ ]:
# TODO: def summary_line(path, device):


# print(summary_line(PLAYS, "phone"))
# print(summary_line(PLAYS, "radio"))
# print(summary_line("data/clean/play.csv", "phone"))

## Part 2: tests

Write each check as a function whose name starts with `test_` and says, as
a sentence, what should be true. At least two per function, including the
awkward case: the empty list, the empty text, the missing file.

One more, which feels backwards: a test that `load_rows` on a missing file
**does** raise `FileNotFoundError`. Write it by hand the way the
walkthrough did, with `try`, `except` and `raise AssertionError(...)`.

In [ ]:
def test_safe_highest_of_a_normal_list():
    assert safe_highest([3, 10, 7]) == 10


# TODO: more tests

The test runner from the walkthrough. Add your test functions to the list
and run it.

In [ ]:
def run_tests(tests):
    """Call every test, report each one, and return how many failed."""
    failed = 0
    for test in tests:
        try:
            test()
            print(f"PASSED  {test.__name__}")
        except AssertionError:
            print(f"FAILED  {test.__name__}")
            failed += 1
    print(f"{len(tests) - failed} passed, {failed} failed")
    return failed


run_tests([
    test_safe_highest_of_a_normal_list,
    # TODO: add yours
])

## Now as a real test file

Copy your three functions into `sessions/session-12/exercises/robust.py`,
and your tests into `sessions/session-12/exercises/test_robust.py`. In the
file version, write the missing-file test with `pytest.raises` instead of
by hand.

Then run, in a terminal at the repository root:

```
python -m pytest sessions/session-12/exercises
```

or run the cell below, which does the same thing if pytest is installed.

In [ ]:
if importlib.util.find_spec("pytest") is None:
    print("pytest is not installed in this Python:  python -m pip install pytest")
else:
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "sessions/session-12/exercises",
         "-p", "no:cacheprovider"],
        capture_output=True, text=True,
    )
    print(result.stdout)

## If you have time

Break one of your functions on purpose, say by catching the wrong error in
`safe_highest`, and run the tests again. Does a test catch it? If not, the
test you are missing is the one to write next.